# Solutions — Functions, Return Values & Exceptions

Exercise notebook: `01_python_fundamentals/04_functions_and_exceptions.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

## Exercise 1 — 🟢 Easy · Safe integer conversion

**Working solution**

In [ ]:
def safe_int(value, default=None):
    if isinstance(value, bool):
        return default
    if isinstance(value, int):
        return value
    try:
        return int(str(value).strip())
    except (TypeError, ValueError):
        return default

**Explanation**

EAFP: just try `int()` and catch the specific failures. `int(' 42 ')` already tolerates whitespace; `int('3.0')` raises, which is what we want.

**Complexity:** O(n) in string length.

**Common mistakes**

- Bare `except:` — it also swallows `KeyboardInterrupt`.
- Checking `value.isdigit()` first — misses negative numbers.

## Exercise 2 — 🟡 Medium · Collect validation errors

**Working solution**

In [ ]:
def validate_user(record: dict) -> list[str]:
    errors = []
    if not str(record.get("name") or "").strip():
        errors.append("name is required")
    email = record.get("email")
    if not isinstance(email, str) or email.count("@") != 1:
        errors.append("email is invalid")
    age = record.get("age")
    if not (isinstance(age, int) and not isinstance(age, bool) and 18 <= age <= 120):
        errors.append("age must be an integer between 18 and 120")
    return errors

**Explanation**

Returning a list of errors (instead of raising at the first one) is the right shape for data validation:
users want every problem in one report. An empty list is falsy, so callers write `if errors:`.

**Complexity:** O(1).

**Common mistakes**

- `record['name']` → `KeyError` on missing keys; use `.get`.
- Returning `True/False` and losing the reasons.

## Exercise 3 — 🟡 Medium · Parse records, separating good and bad

**Working solution**

In [ ]:
def parse_records(lines):
    good, bad = [], []
    for line_no, line in enumerate(lines, start=1):
        if not line.strip():
            continue
        fields = [f.strip() for f in line.split(",")]
        if len(fields) != 3:
            bad.append((line_no, f"expected 3 fields, got {len(fields)}"))
            continue
        raw_id, name, raw_amount = fields
        try:
            good.append({"id": int(raw_id), "name": name, "amount": float(raw_amount)})
        except ValueError as exc:
            bad.append((line_no, str(exc)))
    return good, bad

**Explanation**

`enumerate(lines, start=1)` gives human line numbers. The `try` wraps only the conversions, so unrelated bugs aren't swallowed. The caller unpacks with `good, bad = ...`.

**Complexity:** O(n).

**Common mistakes**

- Wrapping the whole loop body in `try/except Exception`.
- Returning a Java-style result object when a tuple is clearer.

## Exercise 4 — 🟠 Hard · Custom exceptions with context and chaining

**Working solution**

In [ ]:
class RecordValidationError(ValueError):
    def __init__(self, field, message, value=None):
        self.field = field
        self.message = message
        self.value = value
        text = f"field {field!r}: {message}"
        if value is not None:
            text += f" (got {value!r})"
        super().__init__(text)


def validate_order(order: dict) -> dict:
    order_id = str(order.get("order_id") or "").strip()
    if not order_id:
        raise RecordValidationError("order_id", "is required")
    raw_qty = order.get("quantity")
    try:
        quantity = int(raw_qty)
    except (TypeError, ValueError) as exc:
        raise RecordValidationError("quantity", "must be an integer", raw_qty) from exc
    if quantity <= 0:
        raise RecordValidationError("quantity", "must be positive", quantity)
    raw_price = order.get("unit_price")
    try:
        price = float(raw_price)
    except (TypeError, ValueError):
        price = -1.0
    if price < 0:
        raise RecordValidationError("unit_price", "must be a non-negative number", raw_price)
    return {"order_id": order_id, "quantity": quantity, "unit_price": price}

**Explanation**

Subclassing `ValueError` means existing `except ValueError` handlers still work — pick the closest built-in parent.
`raise X from exc` preserves the root cause in the traceback ("The above exception was the direct cause..."), the
equivalent of Java's `new MyException(msg, cause)`.

**Java habit to avoid: checked-exception style wrapping everything**

In [ ]:
class ValidationException(Exception):     # unrelated to ValueError -> callers can't catch it generically
    def __init__(self, msg):
        self.msg = msg
    def getMessage(self):                 # getters are unnecessary; use attributes
        return self.msg

**Complexity:** O(1).

**Common mistakes**

- Forgetting `super().__init__(text)` — `str(err)` is then empty.
- Using `raise ... from None` when the cause is useful for debugging.

## Challenge — 🔴 Very Hard · Batch processing with a failure budget

**Working solution**

In [ ]:
class BatchFailed(Exception):
    def __init__(self, summary: dict):
        self.summary = summary
        super().__init__(f"{summary['failed']} of {summary['processed'] + summary['failed']} records failed")


def process_batch(records, transform, max_failure_ratio=0.1):
    output, errors = [], []
    for index, record in enumerate(records):
        try:
            output.append(transform(record))
        except Exception as exc:
            errors.append({"index": index, "error_type": type(exc).__name__, "message": str(exc)})
    summary = {"output": output, "processed": len(output), "failed": len(errors), "errors": errors}
    total = len(output) + len(errors)
    if total and len(errors) / total > max_failure_ratio:
        raise BatchFailed(summary)
    return summary

**Explanation**

`except Exception` deliberately excludes `BaseException` subclasses (`KeyboardInterrupt`, `SystemExit`,
`GeneratorExit`) so Ctrl+C still stops a long job. Attaching the summary to the exception lets the caller write the
rejected records somewhere before failing the job — the pattern you'll reuse in every pipeline later.

**Complexity:** O(n).

**Common mistakes**

- `except:` or `except BaseException:` — makes the job impossible to stop.
- Checking the ratio inside the loop and aborting early without a summary.